[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/cnn_visualization.ipynb)

### Visualizing Filters and Feature Maps in Convolutional Neural Networks

Imports and model definitions

In [ ]:
from torchvision.models import AlexNet_Weights
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
from torchvision import models, transforms

# load the model
model = models.alexnet(weights=AlexNet_Weights.IMAGENET1K_V1)
print(model)
# get all the model children as list
model_children = list(model.children())

Visualizing Convolutional Layer Filters


### Exercise 1

Change `layer` value, to visualize each layer of the AlexNet. 

> `layer` is the index in the list `conv_layers`, which only contains the 5 convolutional layers of AlexNet, so it goes from **0 to 4**. Note that it is not the position of the layer in `model.features` (which also contains the ReLU and MaxPool layers), used in Exercise 3.

You should also define the y variable to visualize the convolutional filters. For this, you need to divide the shape of the first element of  `conv_layers`[`layer`].`weight` tensor, by 8 (you should also convert this to integer).

Each filter has the shape (input channels, height, width), and only its first input channel is shown. In the first layer, the filters are applied directly to the pixels, so we can recognize what they detect (edges, stripes, colour blobs). In the deeper layers, each filter combines tens or hundreds of feature maps from the previous layer, so a single 3x3 slice is hard to interpret.

In [ ]:
# counter to keep count of the conv layers
counter = 0 
model_weights = []
conv_layers = []

for i in range(len(model_children)):
    if type(model_children[i]) == nn.Conv2d:
        counter += 1
        model_weights.append(model_children[i].weight)
        conv_layers.append(model_children[i])
    elif type(model_children[i]) == nn.Sequential:
        for child in model_children[i].children():
            if type(child) == nn.Conv2d:
                counter += 1
                model_weights.append(child.weight)
                conv_layers.append(child)
print(f"Total convolutional layers: {counter}")

# take a look at the conv layers and the respective weights
for weight, conv in zip(model_weights, conv_layers):
    print(f"CONV: {conv} ====> SHAPE: {weight.shape}")


plt.figure(figsize=(20, 17))

# Change layer value
layer = ...
assert isinstance(layer, int) and 0 <= layer < len(conv_layers), \
    f"layer must be between 0 and {len(conv_layers) - 1} (index in conv_layers, not in model.features)"
# Define y
y = conv_layers[layer].weight.shape[0] // 8
for i, kernel in enumerate(model_weights[layer]):
    plt.subplot(8, y, i+1)
    # show only the first input channel of each filter
    plt.imshow(kernel[0, :, :].detach(), cmap='gray')
    plt.axis('off')
plt.show()

The filters of the first layer have exactly 3 input channels (R, G and B), so each one can be shown as a small colour image. This shows not only the orientation of the edges each filter detects, but also the colours it responds to.

In [ ]:
# filters of the first convolutional layer: 64 x 3 x 11 x 11
filters = conv_layers[0].weight.detach().clone()
# rescale each filter to [0, 1], so it can be shown as an RGB image
f_min = filters.amin(dim=(1, 2, 3), keepdim=True)
f_max = filters.amax(dim=(1, 2, 3), keepdim=True)
filters = (filters - f_min) / (f_max - f_min)

plt.figure(figsize=(10, 10))
for i, kernel in enumerate(filters):
    plt.subplot(8, 8, i + 1)
    plt.imshow(kernel.permute(1, 2, 0))  # (C, H, W) -> (H, W, C), as expected by matplotlib
    plt.axis('off')
plt.suptitle("Filters of the first convolutional layer (RGB)")
plt.show()

### Exercise 2
Reading an image and defining the transforms

In this exercise, you must use any image of your choice.

In [ ]:
from PIL import Image
import urllib.request

# read and visualize an image
URL = '...'

# Add a User-Agent header to the request
req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})

with urllib.request.urlopen(req) as url:
    # convert to RGB: PNG images may have an alpha channel and others may be grayscale
    img = Image.open(url).convert('RGB')

plt.imshow(img)
plt.show()

# define the transforms
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # ImageNet statistics, used to train AlexNet
])
# apply the transforms
img = transform(img)
print(img.size())
# unsqueeze to add a batch dimension
img = img.unsqueeze(0)
print(img.size())

Passing the Input Image Through Each Convolutional Layer

The image goes through all the layers of `model.features` (convolutions, ReLUs and max poolings), as it happens in the model, and we keep the output of each convolutional layer.

In [ ]:
model.eval()
results = []
x = img
# we are not training, so there is no need to compute gradients
with torch.no_grad():
    for module in model.features:
        x = module(x)
        if isinstance(module, nn.Conv2d):
            results.append(x)
outputs = results

Visualizing the Feature Maps


In [ ]:
plt.figure(figsize=(30, 30))
layer_viz = outputs[layer][0, :, :, :]
print(layer_viz.size())
for i, feature_map in enumerate(layer_viz):
    plt.subplot(8, y, i + 1)
    plt.imshow(feature_map, cmap='gray')
    plt.axis("off")
plt.show()
plt.close()

### Directly use pre-trained AlexNet for Image Classification and Visualization of the activation maps

Imports and data transformations.

In [ ]:
import torch.nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.models import AlexNet_Weights
import torch.nn.functional as F 
import torchvision.utils as utils
import cv2 
import matplotlib.pyplot as plt
import numpy as np 
from PIL import Image
import urllib.request


data_transforms = transforms.Compose([
    transforms.Resize((224,224)),       # resize the input to 224x224
    transforms.ToTensor(),              # put the input to tensor format
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # normalize the input
])

Image transformations.

In [ ]:
# Use your image

# Add a User-Agent header to the request
req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})

with urllib.request.urlopen(req) as url:
    img = Image.open(url).convert('RGB')

print("original image's shape: " + str(img.size))
# pre-process the input
transformed_img = data_transforms(img)
print("transformed image's shape: " + str(transformed_img.shape))
# form a batch with only one image
batch_img = torch.unsqueeze(transformed_img, 0)
print("image batch's shape: " + str(batch_img.shape))

# load pre-trained AlexNet model
print("\nfeed the input into the pre-trained alexnet to get the output")
weights = AlexNet_Weights.IMAGENET1K_V1
alexnet = models.alexnet(weights=weights)
# put the model to eval mode for testing
alexnet.eval()

# obtain the output of the model
with torch.no_grad():
    output = alexnet(batch_img)
print("output vector's shape: " + str(output.shape))

# show the 5 most probable ImageNet classes
probabilities = F.softmax(output[0], dim=0)
top5 = torch.topk(probabilities, 5)
for prob, idx in zip(top5.values, top5.indices):
    print(f"{weights.meta['categories'][idx]:30s} {prob.item() * 100:.2f}%")

print(alexnet)

### Exercise 3

Visualize activation maps.

In this exercise you need to define the variables `l1`, `l2`, `l3`, `l4` and `l5` (convolutional layer 1, convolutional layer 2, convolutional layer 3, convolutional layer 4 and convolutional layer 5). Note that these values should correspond to the index of each convolutional layer of the AlexNet (HINT: Check the layers of the model, printed in the previous cell).

In [ ]:
I = utils.make_grid(batch_img, nrow=1, normalize=True, scale_each=True)
img = I.permute((1, 2, 0)).cpu().numpy()

l1, l2 ,l3 ,l4 ,l5 = ..., ..., ..., ..., ...

conv_results = []
x = batch_img
with torch.no_grad():
    for idx, operation in enumerate(alexnet.features):
        x = operation(x)

        if idx in {l1, l2, l3, l4, l5}:
            conv_results.append(x)

plt.figure(figsize=(20, 5))
for i in range(5):
    conv_result = conv_results[i]
    N, C, H, W = conv_result.size()

    # mean of all the channels (feature maps) of the layer, resized to the image size
    mean_acti_map = torch.mean(conv_result, 1, True)
    mean_acti_map = F.interpolate(mean_acti_map, size=[224,224], mode='bilinear', align_corners=False)

    map_grid = utils.make_grid(mean_acti_map, nrow=1, normalize=True, scale_each=True)
    map_grid = map_grid.permute((1, 2, 0)).mul(255).byte().cpu().numpy()
    map_grid = cv2.applyColorMap(map_grid, cv2.COLORMAP_JET)
    map_grid = cv2.cvtColor(map_grid, cv2.COLOR_BGR2RGB)
    map_grid = np.float32(map_grid) / 255

    visual_acti_map = 0.6 * img + 0.4 * map_grid

    plt.subplot(1, 5, i + 1)
    plt.imshow(visual_acti_map)
    plt.title(f"Conv layer {i + 1} ({C} x {H} x {W})")
    plt.axis('off')
plt.show()